# 02 - Modelo de Machine Learning

Modelo de clasificacion para predecir si un estudiante obtiene puntaje global alto en Saber 11.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "Vista_Resultados_unicos_Saber_11_20260521.csv"
df = pd.read_csv(DATA_PATH, encoding="utf-8")
df["PUNT_GLOBAL"] = pd.to_numeric(df["PUNT_GLOBAL"], errors="coerce")
df = df.dropna(subset=["PUNT_GLOBAL"]).copy()
df["puntaje_alto"] = (df["PUNT_GLOBAL"] >= 300).astype(int)
df.shape

In [ ]:
feature_cols = [
    "PERIODO", "COLE_AREA_UBICACION", "COLE_BILINGUE", "COLE_CALENDARIO",
    "COLE_CARACTER", "COLE_DEPTO_UBICACION", "COLE_GENERO", "COLE_JORNADA",
    "COLE_MCPIO_UBICACION", "COLE_NATURALEZA", "COLE_SEDE_PRINCIPAL",
    "ESTU_DEPTO_RESIDE", "ESTU_GENERO", "ESTU_MCPIO_RESIDE", "ESTU_NACIONALIDAD",
    "ESTU_PAIS_RESIDE", "ESTU_PRIVADO_LIBERTAD", "FAMI_CUARTOSHOGAR",
    "FAMI_EDUCACIONMADRE", "FAMI_EDUCACIONPADRE", "FAMI_ESTRATOVIVIENDA",
    "FAMI_PERSONASHOGAR", "FAMI_TIENEAUTOMOVIL", "FAMI_TIENECOMPUTADOR",
    "FAMI_TIENEINTERNET", "FAMI_TIENELAVADORA"
]
X = df[feature_cols]
y = df["puntaje_alto"]
y.value_counts(normalize=True).round(3)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
cat_cols = [c for c in feature_cols if c != "PERIODO"]
num_cols = ["PERIODO"]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
])

model = RandomForestClassifier(n_estimators=250, max_depth=12, min_samples_leaf=10, class_weight="balanced", random_state=42, n_jobs=1)
pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])
pipeline.fit(X_train, y_train)

In [ ]:
pred = pipeline.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Matriz de confusion:")
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=["No alto", "Alto"], zero_division=0))

## Interpretacion

El modelo es una primera aproximacion. Tiene buen recall para detectar estudiantes con puntaje alto, pero todavia produce falsos positivos. Puede mejorarse probando otros algoritmos, ajustando el umbral o agregando nuevas variables explicativas.